## Importy

In [1]:
import py3Dmol
from Bio.PDB import MMCIFParser, MMCIFIO, Select, Superimposer
from Bio.PDB.Polypeptide import is_aa
from Bio.SeqUtils import seq1
from Bio import Align

Kod jest wykonywany na przykładowym białku z datasetu (A0A8C6VAB1). Aby go wykonać trzeba rozpakować pliki mmCIF w jego folderze.
Więcej o plikach mmCIF [tutaj](https://mmcif.wwpdb.org/docs/user-guide/guide.html)

## Helpery

In [2]:
def load_structure(path, structure_id="ID"):
    loader = MMCIFParser()
    structure = loader.get_structure(structure_id, path)
    return structure


def visualise_structure(structure):
    view = py3Dmol.view(width=800, height=600)
    view.addModel(structure, "mmcif")
    view.setStyle({'cartoon': {'color': 'blue'}})
    view.setStyle({'resn': 'LIG'}, {'stick': {'colorscheme': 'greenCarbon'}})
    view.zoomTo()
    return view


def save_structure(structure, select, out_path):
    io1 = MMCIFIO()
    io1.set_structure(structure)
    io1.save(out_path, select=select)

## Wizualizacja białek

In [5]:
import py3Dmol

af_structure_path = "../data/02_intermediate/proteins/A0A8C6VAB1/AF-A0A8C6VAB1-F1-model_v6.cif"
af_structure_path = "../data/03_primary/matched_protein_pairs/A0A8C6VAB1/PDB.cif"
with open(af_structure_path) as f:
    alphafold_structure = f.read()

visualise_structure(alphafold_structure)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [7]:
pdb_structure_path = "../data/02_intermediate/proteins/A0A8C6VAB1/PDB-8K6E.cif"
with open(pdb_structure_path) as f:
    pdb_structure = f.read()

visualise_structure(pdb_structure)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Matchowanie podobnych łańcuchów w polipeptydach, ekstrakcja par

In [8]:
def get_chains_data(structure):
    """Extracts standard amino acid residues and sequences for all chains."""
    chains_data = {}
    model = structure[0]

    for chain in model:
        residues = []
        sequence = ""
        for res in chain:
            if is_aa(res, standard=True):
                try:
                    # convert aminoacid's name to its respective one letter sequence code
                    seq_char = seq1(res.resname)
                    sequence += seq_char
                    residues.append(res)
                except KeyError:
                    continue
        if sequence:
            chains_data[chain.id] = (residues, sequence)

    return chains_data

class ResidueSelect(Select):
    """Filter to keep only the specifically matched residues."""
    def __init__(self, keep_residues):
        self.keep_ids = {res.get_full_id() for res in keep_residues}

    def accept_residue(self, residue):
        if residue.get_full_id() in self.keep_ids:
            return 1
        return 0

def match_and_trim_all(file1, file2, out_file1, out_file2, match_threshold=0.95):
    parser = MMCIFParser(QUIET=True)

    print("Parsing structures...")
    struct1 = parser.get_structure("struct1", file1)
    struct2 = parser.get_structure("struct2", file2)

    chains1 = get_chains_data(struct1)
    chains2 = get_chains_data(struct2)

    aligner = Align.PairwiseAligner()
    aligner.mode = 'local'
    aligner.open_gap_score = -10
    aligner.extend_gap_score = -1

    keep_res1 = []
    keep_res2 = []
    match_count = 0

    print("Comparing all chains...")
    for id1, (res1, seq1) in chains1.items():
        for id2, (res2, seq2) in chains2.items():

            alignments = aligner.align(seq1, seq2)
            best_alignment = alignments[0]

            max_possible_score = min(len(seq1), len(seq2))
            percent_matched = (best_alignment.score / max_possible_score)
            if percent_matched >= match_threshold:
                match_count += 1
                print(f"Match Found: File 1 (Chain {id1}) <--> File 2 (Chain {id2}) | Score: {best_alignment.score}")

                for (start1, end1), (start2, end2) in zip(best_alignment.aligned[0], best_alignment.aligned[1]):
                    keep_res1.extend(res1[start1:end1])
                    keep_res2.extend(res2[start2:end2])

    if match_count == 0:
        print("No matching chains found between the two files.")
        return

    print(f"\nTrimming structures down to {len(keep_res1)} strictly matched residues...")

    # Save File 1 with only matched pairs
    io1 = MMCIFIO()
    io1.set_structure(struct1)
    io1.save(out_file1, select=ResidueSelect(keep_res1))
    print(f"Saved matched pairs to: {out_file1}")

    # Save File 2 with only matched pairs
    io2 = MMCIFIO()
    io2.set_structure(struct2)
    io2.save(out_file2, select=ResidueSelect(keep_res2))
    print(f"Saved matched pairs to: {out_file2}")

FILE_1 = af_structure_path
FILE_2 = pdb_structure_path

# 2. DEFINE YOUR DESIRED OUTPUT FILES
OUT_FILE_1 = "AF_matched_only.cif"
OUT_FILE_2 = "PDB_matched_only.cif"

match_and_trim_all(FILE_1, FILE_2, OUT_FILE_1, OUT_FILE_2)

Parsing structures...
Comparing all chains...
Match Found: File 1 (Chain A) <--> File 2 (Chain A) | Score: 355.0

Trimming structures down to 367 strictly matched residues...
Saved matched pairs to: AF_matched_only.cif
Saved matched pairs to: PDB_matched_only.cif


## Wizualizacja otrzymanych łańcuchów które są zgodne w 95%

In [ ]:
with open(OUT_FILE_1) as f:
    data1 = f.read()

visualise_structure(data1)

In [ ]:
with open(OUT_FILE_2) as f:
    data2 = f.read()

visualise_structure(data2)

## Dopasowanie + kalkulacja RMSE

In [ ]:
def align_and_compare(ref_file, alt_file):
    parser = MMCIFParser(QUIET=True)

    print("Loading structures...")
    # It is standard practice to treat the experimental structure as the "fixed" reference
    ref_structure = parser.get_structure("Reference", ref_file)
    alt_structure = parser.get_structure("Alternative", alt_file)

    # 1. Extract Alpha-Carbon (CA) atoms for the alignment
    # We use CA atoms because they represent the core backbone of the protein
    ref_atoms = [atom for atom in ref_structure.get_atoms() if atom.get_name() == 'CA']
    alt_atoms = [atom for atom in alt_structure.get_atoms() if atom.get_name() == 'CA']

    # Safety check to ensure our 1-to-1 mapping is perfect
    if len(ref_atoms) != len(alt_atoms):
        print(f"Error: Atom counts do not match! (Ref: {len(ref_atoms)}, Alt: {len(alt_atoms)})")
        print("Ensure you are using the strictly trimmed files from the previous step.")
        return

    print(f"Aligning {len(ref_atoms)} Alpha-Carbon pairs...")

    # 2. Initialize the Superimposer
    super_imposer = Superimposer()

    # Calculate the optimal rotation/translation matrix to minimize the distance between atom pairs
    super_imposer.set_atoms(ref_atoms, alt_atoms)

    # 3. Apply the calculated transformation to ALL atoms in the alternative structure
    # This moves the entire protein (not just the CA atoms) to overlay the reference
    super_imposer.apply(alt_structure.get_atoms())

    # 4. Get the RMSD
    rmsd = super_imposer.rms
    print("\n" + "="*40)
    print(f"Alignment Complete!")
    print(f"RMSD: {rmsd:.3f} Angstroms")
    print("="*40 + "\n")

REFERENCE_FILE = OUT_FILE_1
ALTERNATIVE_FILE = OUT_FILE_2

align_and_compare(REFERENCE_FILE, ALTERNATIVE_FILE)

Interpretacja wyników RMSE:
  - < 1.5A - praktycznie idealne dopasowanie
  - 2-3A - dobre dopasowanie
  - 3-5A - słabsze dopasowanie
  - \> 5A - znaczące różnice strukturalne